In [26]:
import json
import math
import re
from pathlib import Path
import numpy as np
import pandas as pd

In [27]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
json_path = Path("dataset_hospital 2.json")

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

pacientes = pd.DataFrame(data["pacientes"])
citas = pd.DataFrame(data["citas_medicas"])

print("pacientes:", pacientes.shape)
print("citas_medicas:", citas.shape)

pacientes: (5010, 8)
citas_medicas: (9961, 7)


In [28]:
import pandas as pd

col = "fecha_nacimiento"

formatos = pacientes[col].dropna().astype(str).str.strip().apply(
    lambda x: "YYYY-MM-DD" if pd.notna(pd.to_datetime(x, errors="coerce")) and "-" in x and len(x) == 10
    else "DD de MES de YYYY" if " de " in x.lower()
    else "OTRO"
)

print("Formatos encontrados:")
print(formatos.value_counts())

if formatos.nunique() > 1:
    print("\nHay formatos diferentes en fecha_nacimiento.")
else:
    print("\nNo hay un formato en fecha_nacimiento.")

Formatos encontrados:
fecha_nacimiento
YYYY-MM-DD           5006
DD de MES de YYYY       3
OTRO                    1
Name: count, dtype: int64

Hay formatos diferentes en fecha_nacimiento.


In [29]:
HOY = pd.Timestamp.today().normalize()

MESES = {
    "ene": 1, "enero": 1,
    "feb": 2, "febrero": 2,
    "mar": 3, "marzo": 3,
    "abr": 4, "abril": 4,
    "may": 5, "mayo": 5,
    "jun": 6, "junio": 6,
    "jul": 7, "julio": 7,
    "ago": 8, "agosto": 8,
    "sep": 9, "sept": 9, "septiembre": 9,
    "oct": 10, "octubre": 10,
    "nov": 11, "noviembre": 11,
    "dic": 12, "diciembre": 12,
}

def parse_spanish_date(value):
    if pd.isna(value):
        return pd.NaT
    s = str(value).strip()
    d = pd.to_datetime(s, errors="coerce")
    if not pd.isna(d):
        return d.normalize()
    m = re.match(r"(\d{1,2})\s+de\s+([A-Za-záéíóú]+)\s+de\s+(\d{4})", s, flags=re.I)
    if m:
        day = int(m.group(1))
        month_txt = (
            m.group(2)
            .lower()
            .replace("á", "a")
            .replace("é", "e")
            .replace("í", "i")
            .replace("ó", "o")
            .replace("ú", "u")
        )
        month = MESES.get(month_txt)
        year = int(m.group(3))
        if month:
            try:
                return pd.Timestamp(year=year, month=month, day=day)
            except ValueError:
                return pd.NaT

    return pd.NaT

In [30]:
def normalizar_sexo(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    mapa = {"m": "M", "male": "M", "f": "F", "female": "F"}
    return mapa.get(s, "OTRO")

In [31]:
def primer_nombre(nombre):
    if pd.isna(nombre):
        return None
    return (
        str(nombre).split()[0].lower()
        .replace("á", "a").replace("é", "e")
        .replace("í", "i").replace("ó", "o").replace("ú", "u")
    )


In [32]:
EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")

In [33]:
pac = pacientes.copy()
cit = citas.copy()

pac["fecha_nac_parsed"] = pac["fecha_nacimiento"].apply(parse_spanish_date)
pac["edad_calculada"] = ((HOY - pac["fecha_nac_parsed"]).dt.days / 365.2425).apply(
    lambda x: math.floor(x) if pd.notna(x) else np.nan
)
pac["edad_diff"] = pac["edad"] - pac["edad_calculada"]
pac["sexo_norm"] = pac["sexo"].apply(normalizar_sexo)
pac["first_name"] = pac["nombre"].apply(primer_nombre)
pac["email_valid"] = pac["email"].apply(lambda x: bool(EMAIL_RE.match(str(x).strip())) if pd.notna(x) else None)
pac["telefono_digits"] = pac["telefono"].apply(lambda x: re.sub(r"\D", "", str(x)) if pd.notna(x) else None)
pac["telefono_valid_10"] = pac["telefono_digits"].apply(lambda x: len(x) == 10 if x is not None else None)

cit["fecha_cita_parsed"] = pd.to_datetime(cit["fecha_cita"], errors="coerce")

In [34]:
female_names = {"maria", "claudia", "andrea", "laura", "cindy", "isabel", "sandra", "daniela", "patricia", "diana"}
male_names = {"juan", "carlos", "paul", "gabriel", "hermes", "andres", "pablo", "jorge", "oscar", "miguel"}

pac["name_sex_conflict"] = pac.apply(
    lambda r: (
        (r["first_name"] in female_names and r["sexo_norm"] == "M")
        or (r["first_name"] in male_names and r["sexo_norm"] == "F")
    ),
    axis=1,
)

resumen_pacientes = pd.DataFrame({
    "problema": [
        "Registros totales",
        "IDs duplicados",
        "Filas completamente duplicadas",
        "Duplicados por nombre + fecha_nacimiento",
        "fecha_nacimiento inválida",
        "edad faltante",
        "edad inconsistente con fecha_nacimiento",
        "edad inconsistente (diferencia >= 2 años)",
        "sexo faltante",
        "sexo con codificación mixta (M/F vs Male/Female)",
        "sexo potencialmente inconsistente con el nombre",
        "email faltante",
        "email inválido (no nulo)",
        "telefono faltante",
        "telefono inválido (no nulo, distinto de 10 dígitos)",
        "ciudad faltante",
    ],
    "valor": [
        len(pac),
        int(pac["id_paciente"].duplicated().sum()),
        int(pac.duplicated().sum()),
        int(pac.duplicated(subset=["nombre", "fecha_nacimiento"]).sum()),
        int(pac["fecha_nacimiento"].notna().sum() - pac["fecha_nac_parsed"].notna().sum()),
        int(pac["edad"].isna().sum()),
        int(((pac["edad"].notna()) & (pac["edad_calculada"].notna()) & (pac["edad"] != pac["edad_calculada"])).sum()),
        int(((pac["edad"].notna()) & (pac["edad_calculada"].notna()) & ((pac["edad"] - pac["edad_calculada"]).abs() >= 2)).sum()),
        int(pac["sexo"].isna().sum()),
        int(pac["sexo"].dropna().nunique()),
        int(pac["name_sex_conflict"].sum()),
        int(pac["email"].isna().sum()),
        int(((pac["email"].notna()) & (pac["email_valid"] == False)).sum()),
        int(pac["telefono"].isna().sum()),
        int(((pac["telefono"].notna()) & (pac["telefono_valid_10"] == False)).sum()),
        int(pac["ciudad"].isna().sum()),
    ]
})

resumen_pacientes


,problema,valor
0,Registros totales,5010
1,IDs duplicados,10
2,Filas completamente duplicadas,10
3,Duplicados por nombre + fecha_nacimiento,25
4,fecha_nacimiento inválida,1
5,edad faltante,1647
6,edad inconsistente con fecha_nacimiento,2071
7,edad inconsistente (diferencia >= 2 años),1653
8,sexo faltante,1023
9,sexo con codificación mixta (M/F vs Male/Female),4


In [35]:
print("Distribución original de sexo:")
display(pac["sexo"].value_counts(dropna=False).rename_axis("sexo").reset_index(name="conteo"))

print("\nEjemplos de edades inconsistentes:")
display(
    pac.loc[
        (pac["edad"].notna()) & (pac["edad_calculada"].notna()) & ((pac["edad"] - pac["edad_calculada"]).abs() >= 10),
        ["id_paciente", "nombre", "fecha_nacimiento", "edad", "edad_calculada"]
    ].head(10)
)

print("\nEjemplo de fecha_nacimiento inválida:")
display(pac.loc[pac["fecha_nac_parsed"].isna(), ["id_paciente", "nombre", "fecha_nacimiento", "edad"]])

print("\nEjemplos de IDs duplicados:")
display(pac.loc[pac["id_paciente"].duplicated(keep=False), ["id_paciente", "nombre", "fecha_nacimiento"]].head(20))


Distribución original de sexo:


,sexo,conteo
0,Male,1049
1,None,1023
2,F,1004
3,Female,972
4,M,962



Ejemplos de edades inconsistentes:


,id_paciente,nombre,fecha_nacimiento,edad,edad_calculada
3,4,Andrea López,1951-11-18,47.0,74.0
4,5,Juan Gómez,1961-09-05,81.0,64.0
10,11,María López,1961-06-15,23.0,64.0
11,12,Carlos López,1955-03-07,53.0,71.0
12,13,Juan Rojas,2009-10-02,56.0,16.0
17,18,Claudia López,1977-01-24,66.0,49.0
18,19,María Pérez,1984-03-02,13.0,42.0
21,22,Juan Pérez,1977-03-29,34.0,48.0
25,26,Juan Rojas,1994-10-24,80.0,31.0
31,32,María Torres,1962-12-15,50.0,63.0



Ejemplo de fecha_nacimiento inválida:


,id_paciente,nombre,fecha_nacimiento,edad
84,85,Juan Torres,1959-06-33,36.0



Ejemplos de IDs duplicados:


,id_paciente,nombre,fecha_nacimiento
499,500,Andrea Pérez,1975-02-13
999,1000,Carlos Torres,1996-06-21
1499,1500,María Gómez,1967-05-25
1999,2000,Juan López,1987-02-23
2499,2500,Claudia Gómez,1984-06-03
2999,3000,María Torres,1967-04-06
3499,3500,Andrea Gómez,1983-03-24
3999,4000,Claudia López,1976-04-18
4499,4500,Andrea Torres,1978-12-07
4999,5000,Claudia Gómez,1966-03-17


In [36]:
resumen_citas = pd.DataFrame({
    "problema": [
        "Registros totales",
        "IDs de cita duplicados",
        "Filas completamente duplicadas",
        "fecha_cita faltante",
        "fecha_cita inválida (no nula pero no parseable)",
        "especialidad faltante",
        "medico faltante",
        "costo faltante",
        "estado_cita faltante",
        "id_paciente sin correspondencia en pacientes",
        "citas completadas sin costo",
        "citas completadas sin medico",
        "citas completadas sin especialidad",
        "citas canceladas con costo informado",
        "citas reprogramadas con fecha pasada",
    ],
    "valor": [
        len(cit),
        int(cit["id_cita"].duplicated().sum()),
        int(cit.duplicated().sum()),
        int(cit["fecha_cita"].isna().sum()),
        int(cit["fecha_cita"].notna().sum() - cit["fecha_cita_parsed"].notna().sum()),
        int(cit["especialidad"].isna().sum()),
        int(cit["medico"].isna().sum()),
        int(cit["costo"].isna().sum()),
        int(cit["estado_cita"].isna().sum()),
        int((~cit["id_paciente"].isin(pac["id_paciente"])).sum()),
        int(((cit["estado_cita"] == "Completada") & (cit["costo"].isna())).sum()),
        int(((cit["estado_cita"] == "Completada") & (cit["medico"].isna())).sum()),
        int(((cit["estado_cita"] == "Completada") & (cit["especialidad"].isna())).sum()),
        int(((cit["estado_cita"] == "Cancelada") & (cit["costo"].notna())).sum()),
        int(((cit["estado_cita"] == "Reprogramada") & (cit["fecha_cita_parsed"].notna()) & (cit["fecha_cita_parsed"] < HOY)).sum()),
    ]
})

resumen_citas

,problema,valor
0,Registros totales,9961
1,IDs de cita duplicados,0
2,Filas completamente duplicadas,0
3,fecha_cita faltante,3278
4,fecha_cita inválida (no nula pero no parseable),3314
5,especialidad faltante,1673
6,medico faltante,2033
7,costo faltante,1724
8,estado_cita faltante,2542
9,id_paciente sin correspondencia en pacientes,190


In [37]:

print("Distribución original de estado_cita:")
display(cit["estado_cita"].value_counts(dropna=False).rename_axis("estado_cita").reset_index(name="conteo"))

print("\nEjemplos de fecha_cita inválida:")
display(
    cit.loc[
        cit["fecha_cita"].notna() & cit["fecha_cita_parsed"].isna(),
        ["id_cita", "id_paciente", "fecha_cita", "estado_cita"]
    ].head(10)
)

print("\nEjemplos de integridad referencial rota (id_paciente sin maestro):")
display(
    cit.loc[
        ~cit["id_paciente"].isin(pac["id_paciente"]),
        ["id_cita", "id_paciente", "fecha_cita", "especialidad", "estado_cita"]
    ].head(10)
)


Distribución original de estado_cita:


,estado_cita,conteo
0,None,2542
1,Reprogramada,2533
2,Cancelada,2490
3,Completada,2396



Ejemplos de fecha_cita inválida:


,id_cita,id_paciente,fecha_cita,estado_cita
4,1cf4b240-0f86-47a1-bab1-53568b25d786,3,2023-19-01,Reprogramada
5,89cee699-c86e-4169-8764-18e57d4e79ee,3,2023-16-01,Cancelada
6,b3b54baf-abdc-47e0-848c-a256150884e6,3,2023-20-01,Cancelada
8,a673a003-07fb-4b49-a888-f583ca394d25,4,2023-19-01,Reprogramada
14,757c5b9c-ef67-4537-b862-efee137077f5,6,2023-18-01,Completada
18,6b2a9862-9349-4ef0-a183-e6b726d153da,8,2023-19-01,None
19,2b9461b6-05fb-4086-9c58-df5ede8cfad1,9,2023-13-01,Cancelada
21,eea9ed6e-93bf-4afc-8894-b461735079b4,10,2023-17-01,Cancelada
28,ba53d833-4979-4e7e-86e3-605b7b1b8978,14,2023-13-01,Completada
31,62b59a80-a513-40b8-b611-e46d02f97958,15,2023-14-01,Reprogramada



Ejemplos de integridad referencial rota (id_paciente sin maestro):


,id_cita,id_paciente,fecha_cita,especialidad,estado_cita
183,ff335846-cf4a-427c-a598-4d0c49019fcf,6416,None,Ginecología,None
232,9ceaa50a-16dd-431e-8345-c5e4a32e7c6d,6148,None,None,Cancelada
279,86fbd10f-c3ef-490c-a4f7-0ab55e4745fe,6141,2022-12-08,None,Reprogramada
297,6f46ce42-8e1f-4ed8-b80a-cdb1f238a715,6095,None,Neurología,Cancelada
338,3f08c4ff-37c8-41e1-bea8-1e59172f0b6a,6000,2025-04-06,Ortopedia,Reprogramada
356,64bdd3cd-8a7b-48cc-8123-227f3ed4d23e,6327,2023-11-29,Pediatría,Cancelada
546,e6111e33-36b3-4065-92f1-68068598a3be,6157,2025-07-30,Ginecología,None
821,2933d6a4-e3b8-46dd-a3b8-85a2164adf32,6065,None,Pediatría,Reprogramada
928,de2dcdd7-50ad-4494-8772-34d91110b10a,6037,None,Neurología,None
967,b9c18dc9-51ce-45e2-b502-71fca15cf122,6477,2023-14-01,Pediatría,Completada


In [38]:
import json
import math
import re
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

HOY = pd.Timestamp.today().normalize()

MESES = {
    "ene": 1, "enero": 1,
    "feb": 2, "febrero": 2,
    "mar": 3, "marzo": 3,
    "abr": 4, "abril": 4,
    "may": 5, "mayo": 5,
    "jun": 6, "junio": 6,
    "jul": 7, "julio": 7,
    "ago": 8, "agosto": 8,
    "sep": 9, "sept": 9, "septiembre": 9,
    "oct": 10, "octubre": 10,
    "nov": 11, "noviembre": 11,
    "dic": 12, "diciembre": 12,
}

EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")

female_names = {"maria", "claudia", "andrea", "laura", "cindy", "isabel", "sandra", "daniela", "patricia", "diana"}
male_names = {"juan", "carlos", "paul", "gabriel", "hermes", "andres", "pablo", "jorge", "oscar", "miguel"}

def quitar_tildes(texto):
    if pd.isna(texto):
        return texto
    return "".join(
        c for c in unicodedata.normalize("NFD", str(texto))
        if unicodedata.category(c) != "Mn"
    )

def texto_limpio(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def texto_titulo(s):
    if pd.isna(s):
        return np.nan
    s = texto_limpio(s)
    return s.title() if pd.notna(s) else np.nan

def parse_spanish_date(value):
    if pd.isna(value):
        return pd.NaT
    
    s = str(value).strip()
    
    d = pd.to_datetime(s, errors="coerce")
    if not pd.isna(d):
        return d.normalize()

    m = re.match(r"(\d{1,2})\s+de\s+([A-Za-záéíóúñ]+)\s+de\s+(\d{4})", s, flags=re.I)
    if m:
        day = int(m.group(1))
        month_txt = quitar_tildes(m.group(2).lower())
        month = MESES.get(month_txt)
        year = int(m.group(3))
        if month:
            try:
                return pd.Timestamp(year=year, month=month, day=day)
            except ValueError:
                return pd.NaT
    return pd.NaT

def calcular_edad(fecha_nacimiento, hoy=HOY):
    if pd.isna(fecha_nacimiento):
        return np.nan
    edad = math.floor((hoy - fecha_nacimiento).days / 365.2425)
    if edad < 0 or edad > 120:
        return np.nan
    return edad

def normalizar_sexo(value):
    if pd.isna(value):
        return np.nan
    s = quitar_tildes(str(value).strip().lower())
    mapa = {
        "m": "M", "male": "M", "masculino": "M", "hombre": "M",
        "f": "F", "female": "F", "femenino": "F", "mujer": "F"
    }
    return mapa.get(s, np.nan)

def inferir_sexo_por_nombre(nombre):
    if pd.isna(nombre):
        return np.nan
    first = quitar_tildes(str(nombre).split()[0].lower())
    if first in female_names:
        return "F"
    if first in male_names:
        return "M"
    return np.nan

def normalizar_email(email):
    if pd.isna(email):
        return np.nan
    s = str(email).strip().lower()
    if EMAIL_RE.match(s):
        return s
    return np.nan

def normalizar_telefono(tel):
    if pd.isna(tel):
        return np.nan
    digits = re.sub(r"\D", "", str(tel))
    return digits if len(digits) == 10 else np.nan

def score_completitud_paciente(row):
    cols = ["nombre", "fecha_nacimiento", "edad", "sexo", "email", "telefono", "ciudad"]
    return sum(pd.notna(row[c]) for c in cols)

def score_completitud_cita(row):
    cols = ["fecha_cita", "especialidad", "medico", "costo", "estado_cita"]
    return sum(pd.notna(row[c]) for c in cols)

pac_raw = pacientes.copy()
cit_raw = citas.copy()


print("pac_raw:", pac_raw.shape)
print("cit_raw:", cit_raw.shape)

pac_raw: (5010, 8)
cit_raw: (9961, 7)


In [39]:
pac_clean = pac_raw.copy()

for col in ["nombre", "email", "telefono", "ciudad", "sexo"]:
    if col in pac_clean.columns:
        pac_clean[col] = pac_clean[col].apply(texto_limpio)

pac_clean["nombre"] = pac_clean["nombre"].apply(texto_titulo)
pac_clean["ciudad"] = pac_clean["ciudad"].apply(lambda x: texto_titulo(x) if pd.notna(x) else x)

pac_clean["fecha_nacimiento_original"] = pac_clean["fecha_nacimiento"]
pac_clean["fecha_nacimiento"] = pac_clean["fecha_nacimiento"].apply(parse_spanish_date)

pac_clean["edad_original"] = pac_clean["edad"]
pac_clean["edad_calculada"] = pac_clean["fecha_nacimiento"].apply(calcular_edad)

def reparar_edad(row):
    edad = row["edad_original"]
    edad_calc = row["edad_calculada"]
    
    if pd.isna(edad):
        return edad_calc
    
    if edad < 0 or edad > 120:
        return edad_calc
    
    if pd.notna(edad_calc) and abs(edad - edad_calc) >= 2:
        return edad_calc
    
    return edad

pac_clean["edad"] = pac_clean.apply(reparar_edad, axis=1)

pac_clean["sexo_original"] = pac_clean["sexo"]
pac_clean["sexo"] = pac_clean["sexo"].apply(normalizar_sexo)


pac_clean["sexo_inferido_nombre"] = pac_clean["nombre"].apply(inferir_sexo_por_nombre)
pac_clean["sexo"] = pac_clean["sexo"].fillna(pac_clean["sexo_inferido_nombre"])


pac_clean["email_original"] = pac_clean["email"]
pac_clean["email"] = pac_clean["email"].apply(normalizar_email)


pac_clean["telefono_original"] = pac_clean["telefono"]
pac_clean["telefono"] = pac_clean["telefono"].apply(normalizar_telefono)


pac_clean["ciudad_original"] = pac_clean["ciudad"]
pac_clean["ciudad"] = pac_clean["ciudad"].fillna("DESCONOCIDA")


for col in ["nombre", "email", "telefono", "ciudad", "sexo"]:
    pac_clean[col] = pac_clean[col].replace({"": np.nan, "Nan": np.nan, "None": np.nan})

display(pac_clean.head())

,id_paciente,nombre,fecha_nacimiento,edad,sexo,email,telefono,ciudad,fecha_nacimiento_original,edad_original,edad_calculada,sexo_original,sexo_inferido_nombre,email_original,telefono_original,ciudad_original
0,1,Claudia Torres,1954-01-08,72.0,F,user1@example.com,3429501064,Barranquilla,1954-01-08,NaN,72.0,Female,F,user1@example.com,342-950-1064,Barranquilla
1,2,Carlos Gómez,1965-01-01,61.0,F,NaN,NaN,Cali,1965-01-01,58.0,61.0,Female,M,NaN,NaN,Cali
2,3,Carlos Gómez,2009-03-08,16.0,M,user3@example.com,3157898999,Bucaramanga,2009-03-08,16.0,17.0,NaN,M,user3@example.com,3157898999,Bucaramanga
3,4,Andrea López,1951-11-18,74.0,F,user4@example.com,NaN,Barranquilla,1951-11-18,47.0,74.0,F,F,user4@example.com,NaN,Barranquilla
4,5,Juan Gómez,1961-09-05,64.0,F,user5@example.com,NaN,Bucaramanga,1961-09-05,81.0,64.0,Female,M,user5@example.com,NaN,Bucaramanga


In [40]:
pac_clean = pac_clean.drop_duplicates().copy()

if pac_clean["id_paciente"].duplicated().any():
    pac_clean["score_completitud"] = pac_clean.apply(score_completitud_paciente, axis=1)
    pac_clean = (
        pac_clean.sort_values(["id_paciente", "score_completitud"], ascending=[True, False])
                 .drop_duplicates(subset=["id_paciente"], keep="first")
                 .drop(columns=["score_completitud"])
                 .copy()
    )

if pac_clean.duplicated(subset=["nombre", "fecha_nacimiento"]).any():
    pac_clean["score_completitud"] = pac_clean.apply(score_completitud_paciente, axis=1)
    pac_clean = (
        pac_clean.sort_values(["nombre", "fecha_nacimiento", "score_completitud"], ascending=[True, True, False])
                 .drop_duplicates(subset=["nombre", "fecha_nacimiento"], keep="first")
                 .drop(columns=["score_completitud"])
                 .copy()
    )

pac_clean = pac_clean.reset_index(drop=True)

print("Pacientes después de deduplicación:", pac_clean.shape)
print("IDs duplicados restantes:", pac_clean["id_paciente"].duplicated().sum())
print("Duplicados nombre+fecha restantes:", pac_clean.duplicated(subset=["nombre", "fecha_nacimiento"]).sum())

Pacientes después de deduplicación: (4985, 16)
IDs duplicados restantes: 0
Duplicados nombre+fecha restantes: 0


In [41]:
pac_clean.loc[
    (pac_clean["fecha_nacimiento"].notna()) &
    ((pac_clean["fecha_nacimiento"] > HOY) | (pac_clean["fecha_nacimiento"] < pd.Timestamp("1900-01-01"))),
    "fecha_nacimiento"
] = pd.NaT

pac_clean["edad_calculada"] = pac_clean["fecha_nacimiento"].apply(calcular_edad)
pac_clean["edad"] = pac_clean["edad"].where(pac_clean["edad"].notna(), pac_clean["edad_calculada"])

mask_edad_diff = (
    pac_clean["edad"].notna() &
    pac_clean["edad_calculada"].notna() &
    ((pac_clean["edad"] - pac_clean["edad_calculada"]).abs() >= 2)
)
pac_clean.loc[mask_edad_diff, "edad"] = pac_clean.loc[mask_edad_diff, "edad_calculada"]

cols_pac_final = [
    "id_paciente", "nombre", "fecha_nacimiento", "edad", "sexo",
    "email", "telefono", "ciudad"
]
pac_final = pac_clean[cols_pac_final].copy()

print("Dataset final de pacientes:")
display(pac_final.head(10))

Dataset final de pacientes:


,id_paciente,nombre,fecha_nacimiento,edad,sexo,email,telefono,ciudad
0,4661,Andrea Gómez,1950-01-18,76.0,F,user4661@example.com,3192637519,Barranquilla
1,4615,Andrea Gómez,1950-01-22,75.0,M,NaN,NaN,DESCONOCIDA
2,2592,Andrea Gómez,1950-07-23,75.0,F,NaN,3587298141,DESCONOCIDA
3,366,Andrea Gómez,1950-09-21,75.0,F,user366@example.com,3151911634,DESCONOCIDA
4,3036,Andrea Gómez,1950-12-15,75.0,F,NaN,3115449442,Bogotá
5,91,Andrea Gómez,1951-02-24,75.0,F,NaN,3715886783,Medellín
6,3549,Andrea Gómez,1951-03-12,75.0,F,NaN,3188271384,Bucaramanga
7,1667,Andrea Gómez,1951-11-16,74.0,F,user1667@example.com,NaN,Medellín
8,2996,Andrea Gómez,1952-04-10,73.0,F,NaN,3427823171,Barranquilla
9,810,Andrea Gómez,1952-08-13,73.0,M,user810@example.com,NaN,Medellín


In [42]:
cit_clean = cit_raw.copy()

for col in ["especialidad", "medico", "estado_cita"]:
    if col in cit_clean.columns:
        cit_clean[col] = cit_clean[col].apply(texto_limpio)

cit_clean["especialidad"] = cit_clean["especialidad"].apply(lambda x: texto_titulo(x) if pd.notna(x) else x)
cit_clean["medico"] = cit_clean["medico"].apply(lambda x: texto_titulo(x) if pd.notna(x) else x)


cit_clean["fecha_cita_original"] = cit_clean["fecha_cita"]
cit_clean["fecha_cita"] = pd.to_datetime(cit_clean["fecha_cita"], errors="coerce")


def normalizar_estado(estado):
    if pd.isna(estado):
        return np.nan
    s = quitar_tildes(str(estado).strip().lower())
    mapa = {
        "completada": "Completada",
        "cancelada": "Cancelada",
        "reprogramada": "Reprogramada",
        "pendiente": "Pendiente",
        "programada": "Programada"
    }
    return mapa.get(s, np.nan)

cit_clean["estado_cita_original"] = cit_clean["estado_cita"]
cit_clean["estado_cita"] = cit_clean["estado_cita"].apply(normalizar_estado)

cit_clean["costo"] = pd.to_numeric(cit_clean["costo"], errors="coerce")


display(cit_clean.head())

,id_cita,id_paciente,fecha_cita,especialidad,medico,costo,estado_cita,fecha_cita_original,estado_cita_original
0,760b31a6-c470-4bf3-add1-878703b403ba,1,NaT,Cardiología,Dr. Juan Valdez,NaN,Reprogramada,None,Reprogramada
1,0810c43c-fe5a-4461-90af-34f6daf8f7b7,1,2022-02-14,Pediatría,Dr. Andrés Murcia,100.0,Completada,2022-02-14,Completada
2,2f38fd84-4db6-48d5-989d-8c7e4a34dcaa,2,NaT,Neurología,NaN,200.0,Cancelada,None,Cancelada
3,fc660b15-69c7-4de9-a860-9c73c1174350,2,2025-06-16,Ginecología,NaN,120.0,Completada,2025-06-16,Completada
4,1cf4b240-0f86-47a1-bab1-53568b25d786,3,NaT,Cardiología,NaN,NaN,Reprogramada,2023-19-01,Reprogramada


In [43]:
cit_clean = cit_clean.drop_duplicates().copy()

if cit_clean["id_cita"].duplicated().any():
    cit_clean["score_completitud"] = cit_clean.apply(score_completitud_cita, axis=1)
    cit_clean = (
        cit_clean.sort_values(["id_cita", "score_completitud"], ascending=[True, False])
                 .drop_duplicates(subset=["id_cita"], keep="first")
                 .drop(columns=["score_completitud"])
                 .copy()
    )

cit_clean.loc[cit_clean["estado_cita"] == "Cancelada", "costo"] = np.nan

mediana_por_especialidad = cit_clean.groupby("especialidad")["costo"].median()

mask_completada_sin_costo = (
    (cit_clean["estado_cita"] == "Completada") &
    (cit_clean["costo"].isna()) &
    (cit_clean["especialidad"].notna())
)

cit_clean.loc[mask_completada_sin_costo, "costo"] = (
    cit_clean.loc[mask_completada_sin_costo, "especialidad"].map(mediana_por_especialidad)
)

mediana_global = cit_clean["costo"].median()
mask_completada_sin_costo2 = (
    (cit_clean["estado_cita"] == "Completada") &
    (cit_clean["costo"].isna())
)
cit_clean.loc[mask_completada_sin_costo2, "costo"] = mediana_global

cit_clean.loc[(cit_clean["estado_cita"] == "Completada") & (cit_clean["medico"].isna()), "medico"] = "NO INFORMADO"
cit_clean.loc[(cit_clean["estado_cita"] == "Completada") & (cit_clean["especialidad"].isna()), "especialidad"] = "NO INFORMADA"

cit_clean.loc[
    (cit_clean["estado_cita"] == "Reprogramada") &
    (cit_clean["fecha_cita"].notna()) &
    (cit_clean["fecha_cita"] < HOY),
    "estado_cita"
] = "Pendiente Revision"

cit_clean["especialidad"] = cit_clean["especialidad"].fillna("NO INFORMADA")
cit_clean["medico"] = cit_clean["medico"].fillna("NO INFORMADO")

In [44]:
cit_huerfanas = cit_clean.loc[~cit_clean["id_paciente"].isin(pac_final["id_paciente"])].copy()
cit_final = cit_clean.loc[cit_clean["id_paciente"].isin(pac_final["id_paciente"])].copy()

print("Citas válidas:", cit_final.shape)
print("Citas huérfanas enviadas a cuarentena:", cit_huerfanas.shape)

if len(cit_huerfanas) > 0:
    print("\nEjemplos de citas huérfanas:")
    display(cit_huerfanas.head(10))

Citas válidas: (9744, 9)
Citas huérfanas enviadas a cuarentena: (217, 9)

Ejemplos de citas huérfanas:


,id_cita,id_paciente,fecha_cita,especialidad,medico,costo,estado_cita,fecha_cita_original,estado_cita_original
183,ff335846-cf4a-427c-a598-4d0c49019fcf,6416,NaT,Ginecología,Dra. Lina Torres,NaN,NaN,None,NaN
215,b04744da-a675-4334-9ac7-b49f07e6b52a,109,NaT,Pediatría,Dr. Camilo Rojas,100.0,NaN,None,NaN
232,9ceaa50a-16dd-431e-8345-c5e4a32e7c6d,6148,NaT,NO INFORMADA,Dra. Lina Torres,NaN,Cancelada,None,Cancelada
279,86fbd10f-c3ef-490c-a4f7-0ab55e4745fe,6141,2022-12-08,NO INFORMADA,Dr. Juan Valdez,220.0,Pendiente Revision,2022-12-08,Reprogramada
297,6f46ce42-8e1f-4ed8-b80a-cdb1f238a715,6095,NaT,Neurología,Dr. Camilo Rojas,NaN,Cancelada,None,Cancelada
338,3f08c4ff-37c8-41e1-bea8-1e59172f0b6a,6000,2025-04-06,Ortopedia,Dr. Camilo Rojas,120.0,Pendiente Revision,2025-04-06,Reprogramada
356,64bdd3cd-8a7b-48cc-8123-227f3ed4d23e,6327,2023-11-29,Pediatría,Dra. Lina Torres,NaN,Cancelada,2023-11-29,Cancelada
546,e6111e33-36b3-4065-92f1-68068598a3be,6157,2025-07-30,Ginecología,Dra. Lina Torres,220.0,NaN,2025-07-30,NaN
821,2933d6a4-e3b8-46dd-a3b8-85a2164adf32,6065,NaT,Pediatría,NO INFORMADO,200.0,Reprogramada,None,Reprogramada
928,de2dcdd7-50ad-4494-8772-34d91110b10a,6037,NaT,Neurología,Dr. Camilo Rojas,180.0,NaN,None,NaN


In [45]:
resumen_pacientes_limpio = pd.DataFrame({
    "problema": [
        "Registros totales",
        "IDs duplicados",
        "Filas completamente duplicadas",
        "Duplicados por nombre + fecha_nacimiento",
        "fecha_nacimiento faltante o inválida",
        "edad faltante",
        "sexo faltante",
        "email faltante",
        "telefono faltante",
        "ciudad faltante",
    ],
    "valor": [
        len(pac_final),
        int(pac_final["id_paciente"].duplicated().sum()),
        int(pac_final.duplicated().sum()),
        int(pac_final.duplicated(subset=["nombre", "fecha_nacimiento"]).sum()),
        int(pac_final["fecha_nacimiento"].isna().sum()),
        int(pac_final["edad"].isna().sum()),
        int(pac_final["sexo"].isna().sum()),
        int(pac_final["email"].isna().sum()),
        int(pac_final["telefono"].isna().sum()),
        int(pac_final["ciudad"].isna().sum()),
    ]
})

resumen_citas_limpio = pd.DataFrame({
    "problema": [
        "Registros totales",
        "IDs de cita duplicados",
        "Filas completamente duplicadas",
        "fecha_cita faltante",
        "especialidad faltante",
        "medico faltante",
        "costo faltante",
        "estado_cita faltante",
        "id_paciente sin correspondencia",
    ],
    "valor": [
        len(cit_final),
        int(cit_final["id_cita"].duplicated().sum()),
        int(cit_final.duplicated().sum()),
        int(cit_final["fecha_cita"].isna().sum()),
        int(cit_final["especialidad"].isna().sum()),
        int(cit_final["medico"].isna().sum()),
        int(cit_final["costo"].isna().sum()),
        int(cit_final["estado_cita"].isna().sum()),
        int((~cit_final["id_paciente"].isin(pac_final["id_paciente"])).sum()),
    ]
})

print("Resumen pacientes limpio:")
display(resumen_pacientes_limpio)

print("\nResumen citas limpio:")
display(resumen_citas_limpio)

Resumen pacientes limpio:


,problema,valor
0,Registros totales,4985
1,IDs duplicados,0
2,Filas completamente duplicadas,0
3,Duplicados por nombre + fecha_nacimiento,0
4,fecha_nacimiento faltante o inválida,1
5,edad faltante,0
6,sexo faltante,0
7,email faltante,2489
8,telefono faltante,1657
9,ciudad faltante,0



Resumen citas limpio:


,problema,valor
0,Registros totales,9744
1,IDs de cita duplicados,0
2,Filas completamente duplicadas,0
3,fecha_cita faltante,6445
4,especialidad faltante,0
5,medico faltante,0
6,costo faltante,3313
7,estado_cita faltante,2478
8,id_paciente sin correspondencia,0


In [47]:
output = {
    "pacientes": pac_final.copy(),
    "citas_medicas": cit_final.copy()
}

json_salida = "dataset_hospital_limpio.json"
with open(json_salida, "w", encoding="utf-8") as f:
    json.dump(
        {
            "pacientes": pac_final.assign(
                fecha_nacimiento=pac_final["fecha_nacimiento"].astype("string")
            ).replace({np.nan: None}).to_dict(orient="records"),
            "citas_medicas": cit_final.assign(
                fecha_cita=cit_final["fecha_cita"].astype("string")
            ).replace({np.nan: None}).to_dict(orient="records")
        },
        f,
        ensure_ascii=False,
        indent=2
    )

cit_huerfanas.to_csv("citas_huerfanas_revision.csv", index=False, encoding="utf-8")
pac_final.to_csv("pacientes_limpio.csv", index=False, encoding="utf-8")
cit_final.to_csv("citas_medicas_limpio.csv", index=False, encoding="utf-8")

print("Archivos generados:")
print("-", json_salida)
print("-", "pacientes_limpio.csv")
print("-", "citas_medicas_limpio.csv")
print("-", "citas_huerfanas_revision.csv")

Archivos generados:
- dataset_hospital_limpio.json
- pacientes_limpio.csv
- citas_medicas_limpio.csv
- citas_huerfanas_revision.csv


In [48]:
comparativo = pd.DataFrame({
    "métrica": [
        "Pacientes totales",
        "Pacientes con ID duplicado",
        "Pacientes con fecha_nacimiento inválida/faltante",
        "Pacientes con edad faltante",
        "Pacientes con sexo faltante",
        "Pacientes con email inválido/faltante",
        "Pacientes con teléfono inválido/faltante",
        "Citas totales",
        "Citas con id_cita duplicado",
        "Citas con fecha inválida/faltante",
        "Citas huérfanas",
        "Citas con costo faltante",
    ],
    "antes": [
        len(pac_raw),
        int(pac_raw["id_paciente"].duplicated().sum()),
        int(pac["fecha_nac_parsed"].isna().sum()),
        int(pac_raw["edad"].isna().sum()),
        int(pac_raw["sexo"].isna().sum()),
        int(pac_raw["email"].isna().sum() + ((pac["email"].notna()) & (pac["email_valid"] == False)).sum()),
        int(pac_raw["telefono"].isna().sum() + ((pac["telefono"].notna()) & (pac["telefono_valid_10"] == False)).sum()),
        len(cit_raw),
        int(cit_raw["id_cita"].duplicated().sum()),
        int(cit["fecha_cita_parsed"].isna().sum()),
        int((~cit_raw["id_paciente"].isin(pac_raw["id_paciente"])).sum()),
        int(cit_raw["costo"].isna().sum()),
    ],
    "después": [
        len(pac_final),
        int(pac_final["id_paciente"].duplicated().sum()),
        int(pac_final["fecha_nacimiento"].isna().sum()),
        int(pac_final["edad"].isna().sum()),
        int(pac_final["sexo"].isna().sum()),
        int(pac_final["email"].isna().sum()),
        int(pac_final["telefono"].isna().sum()),
        len(cit_final),
        int(cit_final["id_cita"].duplicated().sum()),
        int(cit_final["fecha_cita"].isna().sum()),
        int((~cit_final["id_paciente"].isin(pac_final["id_paciente"])).sum()),
        int(cit_final["costo"].isna().sum()),
    ]
})

comparativo["mejora"] = comparativo["antes"] - comparativo["después"]

display(comparativo)

,métrica,antes,después,mejora
0,Pacientes totales,5010,4985,25
1,Pacientes con ID duplicado,10,0,10
2,Pacientes con fecha_nacimiento inválida/faltante,1,1,0
3,Pacientes con edad faltante,1647,0,1647
4,Pacientes con sexo faltante,1023,0,1023
5,Pacientes con email inválido/faltante,2506,2489,17
6,Pacientes con teléfono inválido/faltante,1668,1657,11
7,Citas totales,9961,9744,217
8,Citas con id_cita duplicado,0,0,0
9,Citas con fecha inválida/faltante,6592,6445,147


In [ ]:
import os
from pathlib import Path
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

project_id = "alien-program-491022-g3"
dataset_id = "hospital"


ruta_credenciales = Path(r"C:\Users\JUAN CARLOS HURTADO\Desktop\xpertgroup\alien-program-491022-g3-219e4bef310b.json")


base_path = Path.cwd()

ruta_pacientes = base_path / "pacientes_limpio.csv"
ruta_citas = base_path / "citas_medicas_limpio.csv"


print("-", ruta_pacientes)
print("-", ruta_citas)

credentials = service_account.Credentials.from_service_account_file(
    str(ruta_credenciales)
)

client = bigquery.Client(
    project=project_id,
    credentials=credentials
)
pac_bq = pd.read_csv(ruta_pacientes, encoding="utf-8")
cit_bq = pd.read_csv(ruta_citas, encoding="utf-8")

pac_bq["id_paciente"] = pd.to_numeric(pac_bq["id_paciente"], errors="coerce").astype("Int64")
pac_bq["nombre"] = pac_bq["nombre"].astype("string")
pac_bq["fecha_nacimiento"] = pd.to_datetime(pac_bq["fecha_nacimiento"], errors="coerce").dt.date
pac_bq["edad"] = pd.to_numeric(pac_bq["edad"], errors="coerce").astype("Int64")
pac_bq["sexo"] = pac_bq["sexo"].astype("string")
pac_bq["email"] = pac_bq["email"].astype("string")
pac_bq["telefono"] = pac_bq["telefono"].astype("string")
pac_bq["ciudad"] = pac_bq["ciudad"].astype("string")


cit_bq["id_cita"] = pd.to_numeric(cit_bq["id_cita"], errors="coerce").astype("Int64")
cit_bq["id_paciente"] = pd.to_numeric(cit_bq["id_paciente"], errors="coerce").astype("Int64")
cit_bq["fecha_cita"] = pd.to_datetime(cit_bq["fecha_cita"], errors="coerce")
cit_bq["especialidad"] = cit_bq["especialidad"].astype("string")
cit_bq["medico"] = cit_bq["medico"].astype("string")
cit_bq["costo"] = pd.to_numeric(cit_bq["costo"], errors="coerce").astype("float64")
cit_bq["estado_cita"] = cit_bq["estado_cita"].astype("string")


pac_bq = pac_bq.where(pd.notnull(pac_bq), None)
cit_bq = cit_bq.where(pd.notnull(cit_bq), None)

print("\nPacientes:")
print(pac_bq.dtypes)
print("\nCitas:")
print(cit_bq.dtypes)

dataset_ref = bigquery.Dataset(f"{project_id}.{dataset_id}")
dataset_ref.location = "US"
client.create_dataset(dataset_ref, exists_ok=True)

print(f"\nDataset verificado/creado: {project_id}.{dataset_id}")

job_config_pacientes = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=[
        bigquery.SchemaField("id_paciente", "INT64", mode="NULLABLE"),
        bigquery.SchemaField("nombre", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("fecha_nacimiento", "DATE", mode="NULLABLE"),
        bigquery.SchemaField("edad", "INT64", mode="NULLABLE"),
        bigquery.SchemaField("sexo", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("email", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("telefono", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("ciudad", "STRING", mode="NULLABLE"),
    ],
)

job_config_citas = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=[
        bigquery.SchemaField("id_cita", "INT64", mode="NULLABLE"),
        bigquery.SchemaField("id_paciente", "INT64", mode="NULLABLE"),
        bigquery.SchemaField("fecha_cita", "DATETIME", mode="NULLABLE"),
        bigquery.SchemaField("especialidad", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("medico", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("costo", "FLOAT64", mode="NULLABLE"),
        bigquery.SchemaField("estado_cita", "STRING", mode="NULLABLE"),
    ],
)

tabla_pacientes = f"{project_id}.{dataset_id}.pacientes"

job_pacientes = client.load_table_from_dataframe(
    pac_bq,
    tabla_pacientes,
    job_config=job_config_pacientes
)
job_pacientes.result()


tabla_citas = f"{project_id}.{dataset_id}.citas_medicas"

job_citas = client.load_table_from_dataframe(
    cit_bq,
    tabla_citas,
    job_config=job_config_citas
)
job_citas.result()

tabla_pacientes_obj = client.get_table(tabla_pacientes)
tabla_citas_obj = client.get_table(tabla_citas)

print(f"Pacientes: {tabla_pacientes_obj.num_rows} filas")
print(f"Citas médicas: {tabla_citas_obj.num_rows} filas")

Archivos encontrados correctamente:
- c:\Users\JUAN CARLOS HURTADO\Desktop\xpertgroup\pacientes_limpio.csv
- c:\Users\JUAN CARLOS HURTADO\Desktop\xpertgroup\citas_medicas_limpio.csv

Tipos listos para BigQuery:

Pacientes:
id_paciente                  Int64
nombre              string[python]
fecha_nacimiento            object
edad                         Int64
sexo                string[python]
email               string[python]
telefono            string[python]
ciudad              string[python]
dtype: object

Citas:
id_cita                          Int64
id_paciente                      Int64
fecha_cita              datetime64[ns]
especialidad            string[python]
medico                  string[python]
costo                          float64
estado_cita             string[python]
fecha_cita_original             object
estado_cita_original            object
dtype: object

Dataset verificado/creado: alien-program-491022-g3.hospital

Datos cargados a BigQuery correctamente.
Pacient